# 02 Historical control

Trailing 20-observation mean squared return as the non-learned baseline.

## Setup

In [1]:
# Colab setup: clone the private repo, install deps, mount Drive for artifacts.
# Requires a Colab Secret named GITHUB_TOKEN (fine-grained, Contents read-only on this repo).
import base64, os, subprocess, sys
from google.colab import drive, userdata

REPO = "BerkayPolat-lab/gru-currency-volatility-distillation"
REPO_DIR = "/content/gru-currency-volatility-distillation"

def git(*args):
    # Pass the token as an HTTP header through env vars so it never appears in argv,
    # exception messages, or .git/config, and redact it from any error output.
    token = userdata.get("GITHUB_TOKEN").strip()
    basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env = {**os.environ, "GIT_TERMINAL_PROMPT": "0", "GIT_CONFIG_COUNT": "1",
           "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
           "GIT_CONFIG_VALUE_0": f"Authorization: Basic {basic}"}
    result = subprocess.run(["git", *args], env=env, capture_output=True, text=True)
    if result.returncode != 0:
        stderr = result.stderr.replace(token, "***").replace(basic, "***")
        raise RuntimeError(f"git {args[0]} failed:\n{stderr}") from None

if not os.path.exists(REPO_DIR):
    git("clone", "-q", f"https://github.com/{REPO}.git", REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/model/requirements.txt"], check=True)
sys.path.insert(0, f"{REPO_DIR}/model/src")
os.chdir(f"{REPO_DIR}/model")

drive.mount("/content/drive")
ARTIFACTS = "/content/drive/MyDrive/gru_vol_artifacts"
os.makedirs(ARTIFACTS, exist_ok=True)

SecretNotFoundError: Secret GITHUB_TOKEN does not exist.

In [ ]:
import yaml

with open("configs/base.yaml") as f:
    CFG = yaml.safe_load(f)
CFG

## Control forecasts

Compute the trailing-20 forecast for every validation and test example.

In [ ]:
import json
import numpy as np
import pandas as pd

from gru_vol.data import load_prices, log_returns, load_examples
from gru_vol.models import historical_control
from gru_vol.evaluation import mse, high_vol_mse

examples = load_examples()
with open("artifacts/manifests/data_manifest.json") as f:
    MANIFEST = json.load(f)

# guard against a stale processed file
for name, info in MANIFEST["examples"]["splits"].items():
    assert (examples["split"] == name).sum() == info["examples"], name

window = CFG["historical_control"]["window"]
squared_returns = log_returns(load_prices(CFG)) ** 2

examples["hc"] = historical_control(squared_returns, window).reindex(examples.index)
assert examples["hc"].notna().all()

# cross-check against the features notebook 01 already built
if window == 60:
    assert np.allclose(examples["hc"], examples["var_60"])
if window == 20:
    assert np.allclose(examples["hc"], examples["vol_20"] ** 2)

examples[["split", "target", "hc"]].head()

## Fix the high-volatility cutoff

90th percentile of training targets, frozen before any model is evaluated.

In [ ]:
cutoff = MANIFEST["target"]["high_vol_cutoff"]
train_cutoff = examples.loc[examples["split"].eq("train"), "target"].quantile(CFG["evaluation"]["high_vol_quantile"])
assert np.isclose(cutoff, train_cutoff)
print(f"high-vol cutoff: {cutoff:.4e}")

## Metrics

Overall MSE and high-volatility MSE (with qualifying-date count) in original units. These are the reference for every model's MSE gain.

In [ ]:
# constant train-mean forecast as a sanity floor; the control should beat it
const = examples.loc[examples["split"].eq("train"), "target"].mean()

rows = []
for split in ["train", "validation"]:   # test stays untouched until notebook 06
    part = examples[examples["split"].eq(split)]
    hv, n_hv = high_vol_mse(part["target"], part["hc"], cutoff)
    rows.append({
        "split": split,
        "examples": len(part),
        "mse": mse(part["target"], part["hc"]),
        "high_vol_mse": hv,
        "high_vol_examples": n_hv,
        "const_mse": mse(part["target"], np.full(len(part), const)),
    })
metrics = pd.DataFrame(rows).set_index("split")
assert metrics.loc["validation", "mse"] < metrics.loc["validation", "const_mse"]
metrics.style.format({c: "{:.4e}" for c in ["mse", "high_vol_mse", "const_mse"]})

In [ ]:
for folder in ["artifacts/results", f"{ARTIFACTS}/results"]:
    os.makedirs(folder, exist_ok=True)
    examples[["split", "target", "hc"]].to_csv(f"{folder}/control_predictions.csv", index_label="date")
    with open(f"{folder}/control_metrics.json", "w") as f:
        json.dump({"window": window, "high_vol_cutoff": cutoff,
                   "metrics": metrics.to_dict(orient="index")}, f, indent=2)